In [ ]:
import os
import re
import pypdf
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Config
PDF_PATH = "de.pdf" 
MODEL = "deepseek-r1:7b" 
OUTPUT_MD = "de_interview_prep.md"

chapters_list = [
    """Chapter 1. Data Engineering""",
    """Chapter 2. The Data
Engineering Lifecycle""",
    """Chapter 3. Designing Good
Data Architecture""",
    """Chapter 4. Choosing
Technologies Across the Data
Engineering Lifecycle""",
"""Chapter 5. Data Generation in
Source Systems""",
"""Chapter 6. Storage""",
"""Chapter 7. Ingestion""",
"""Chapter 8. Queries, Modeling, and
Transformation""",
"""Chapter 9. Serving Data for
Analytics, Machine Learning,
and Reverse ETL""",
"""Chapter 10. Security and
Privacy""",
"""Chapter 11. The Future of Data
Engineering"""
]

def get_chapter_hierarchy(pdf_path, chapters):
    reader = pypdf.PdfReader(pdf_path)
    hierarchy = []
    
    # We look for "Chapter X" and "X.X" patterns to avoid skipping subsections
    section_pattern = re.compile(r'^(\d+\.\d+)\s+(.*)') 

    for i, title in enumerate(chapters):
        start_page = -1
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if title.lower() in text.lower():
                start_page = page_num
                break
        
        if start_page != -1:
            hierarchy.append({"title": title, "page": start_page, "type": "chapter"})

    return hierarchy

def get_chapter_ranges(pdf_path, chapters):
    """Finds the start and end page for each chapter title."""
    reader = pypdf.PdfReader(pdf_path)
    chapter_pages = []
    
    print("Mapping chapters to page numbers...")
    for title in chapters:
        found_page = -1
        # Search for the title string in the PDF text
        for i, page in enumerate(reader.pages):
            if title.lower() in page.extract_text().lower():
                found_page = i 
                break
        chapter_pages.append({"title": title, "start": found_page})

    # Calculate end pages based on the next chapter's start
    for i in range(len(chapter_pages)):
        if chapter_pages[i]["start"] == -1:
            continue
        if i + 1 < len(chapter_pages) and chapter_pages[i+1]["start"] != -1:
            chapter_pages[i]["end"] = chapter_pages[i+1]["start"]
        else:
            chapter_pages[i]["end"] = len(reader.pages)
            
    return [c for c in chapter_pages if c["start"] != -1]

def extract_text_from_range(pdf_path, start_page, end_page):
    """Extracts text only within the specified page boundaries."""
    text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for i in range(start_page, end_page):
            extracted = reader.pages[i].extract_text()
            if extracted:
                text += extracted + "\n"
    except Exception as e:
        print(f"Error reading range {start_page}-{end_page}: {e}")
    return text

def summarize_chapter(chapter_text, chapter_num, chapter_title):
    prompt_template = """
You are a senior data engineering architect explaining concepts to other engineers.

Task:
Summarize the provided text from "{chapter_title}" into clear, interview‑relevant sections.
Identify distinct sub-topics, but KEEP OUTPUT CONCISE.

For EACH sub-topic, use ONLY this structure:

## [Sub-topic name]

### What & Why
- What it is (1 line, practical).
- Why it exists (real data/platform pain).
- What breaks if ignored (1 concrete impact).

### Design & Trade-offs
- Typical architecture or approach.
- 2–3 key trade-offs (cost, latency, complexity).
- When NOT to use it.

### Tools (if applicable)
- Common tools/platforms used.
- One sentence on why teams choose them.

### Interview Signals
- 2 high‑signal interview questions.
- Short, direct sample answers.

Rules:
- Be concise. No long explanations.
- Examples only if they add clarity.
- Do NOT invent sections not supported by the text.
- Prefer bullet points over paragraphs.

TEXT:
{chunk_text}
"""

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1800,
        chunk_overlap=150,
        separators=["\n### ", "\n## ", "\n\n", "\n"]
    )

    chunks = splitter.split_text(chapter_text)

    chapter_summary = ""
    for i, chunk in enumerate(chunks):
        current_prompt = prompt_template.format(
            chapter_title=chapter_title,
            chunk_text=chunk
        )

        response = ollama.generate(
            model=MODEL,
            prompt=current_prompt,
            options={
                "temperature": 0.2,
                "num_predict": 800  # hard cap on verbosity
            }
        )

        chapter_summary += response["response"] + "\n\n"
        print(f"Processed chunk {i+1}/{len(chunks)} for {chapter_title}")

    return chapter_summary

def _summarize_chapter(chapter_text, chapter_num, chapter_title):
    prompt_template = """
You are a Staff/Principal Data Engineering architect and interviewer. Write like someone who has built and operated production data platforms.

Task: Produce an interview-grade explanation of the text from "{chapter_title}". Treat each distinct sub-topic separately.

For EACH sub-topic you detect, use this structure (omit any subsection that is not supported by the text):

## Section: [Original Name from PDF]

### Core Concepts and Rationale
- What it is: 1–2 lines, in practical engineering language.
- Why it exists: Real data/platform pain it solves.
- What breaks without it: Impact on data quality, reliability, cost, or governance.
- Enterprise example: {{industry}}, {{scale}}, {{constraint}}, {{failure avoided}}, {{measurable outcome}}.
- Anti-example: When this is overkill or harmful.

### Architectures and Design Choices
- Why this design is chosen over simpler alternatives.
- Main trade-offs (cost vs complexity, latency vs correctness, etc.).
- Enterprise example: Include {{scale}}, {{SLA}}, {{compliance/governance}} angle.
- Typical failure modes in real organizations.
- When NOT to use it (team maturity, budget, use case mismatch).
- Interview defense: How a senior engineer would justify this design.

### Tools and Platforms
- Operational problem this tool/platform abstracts.
- Enterprise example: Migration, modernization, multi-team platform use.
- Where it beats alternatives and where it loses.
- Cost / governance risks (compute blow-ups, lineage, secrets, RBAC).
- Common misuse that causes incidents or bad data.

### Best Practices
- Why this practice is non-negotiable at scale.
- Enterprise example: Incident it prevented or fixed.
- Failure mode if ignored (silent corruption, SLA miss, audit failure).
- How senior engineers enforce it (guardrails, templates, policies, CI/CD).

### Interview-Level Insights
- 3 high-signal “why” questions specific to this sub-topic.
- Concise, strong sample answers that reference trade-offs and constraints.
- Red flags interviewers look for in answers.

TEXT TO ANALYZE:
{chunk_text}

Guidance:
- If the sub-topic is conceptual, emphasize rationale and best practices.
- If it is technical, emphasize architectures and tools.
- Do NOT collapse the whole text into one section; keep sub-topics separate.
- Make questions and answers specific to each sub-topic, not generic.
"""

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000, 
        chunk_overlap=300,
        separators=["\n### ", "\n## ", "\n\n", "\n"]
    )
    chunks = splitter.split_text(chapter_text)
    
    chapter_summary = ""
    for i, chunk in enumerate(chunks):
        # We only pass chapter_title and chunk_text. 
        # Double braces {{ }} are for Ollama to handle.
        current_prompt = prompt_template.format(
            chapter_title=chapter_title,
            chunk_text=chunk
        )
        
        response = ollama.generate(model=MODEL, prompt=current_prompt)
        chapter_summary += response['response'] + "\n\n"
        print(f"Processed chunk {i+1}/{len(chunks)} for {chapter_title}")

    return chapter_summary


def generate_mocks(summary):
    """Maintains your exact mock interview prompt."""
    # YOUR ORIGINAL PROMPT
    prompt = f"""
    Based on this chapter summary, generate EXACTLY 3 mock Data Engineering interview questions (behavioral/technical). 
    For each: Question, then curated acceptable answer (concise, eloquent, 150-250 words).
    
    Format as:
    ### Mock Question 1
    **Q:** [Question]
    **A:** [Answer]
    
    Output ONLY markdown.
    """
    # Truncate to ensure the prompt doesn't exceed LLM limits
    response = ollama.generate(model=MODEL, prompt=prompt.replace("this chapter summary", summary[:8000]))
    return response['response']

# Main execution
if __name__ == "__main__":
    if not os.path.exists(PDF_PATH):
        print(f"File not found: {PDF_PATH}")
    else:
        # 1. Map the chapters to pages
        ranges = get_chapter_ranges(PDF_PATH, chapters_list)
        print(ranges)
        md_content = "# Data Engineering Interview Prep\n\n"
        
        # 2. Process each range
        for i, item in enumerate(ranges, 1):
            print(f"Processing {item['title']} (Pages {item['start']} to {item['end']})...")
            
            #Extract only the relevant text
            chapter_text = extract_text_from_range(PDF_PATH, item['start'], item['end'])
            
            if not chapter_text.strip():
                continue

            # 3. Summarize and Mock
            chap_summary = summarize_chapter(chapter_text, i, item['title'])
            mocks = generate_mocks(chap_summary)
            print(chap_summary)

            md_content += f"{chap_summary}\n\n{mocks}\n\n---\n\n"
            
            # Incremental save
            with open(OUTPUT_MD, 'w', encoding='utf-8') as f:
                f.write(md_content)

        print(f"Done! Created: {OUTPUT_MD}")

Mapping chapters to page numbers...
[{'title': 'Chapter 1. Data Engineering', 'start': 17, 'end': 59}, {'title': 'Chapter 2. The Data\nEngineering Lifecycle', 'start': 59, 'end': 110}, {'title': 'Chapter 3. Designing Good\nData Architecture', 'start': 110, 'end': 170}, {'title': 'Chapter 4. Choosing\nTechnologies Across the Data\nEngineering Lifecycle', 'start': 170, 'end': 224}, {'title': 'Chapter 5. Data Generation in\nSource Systems', 'start': 224, 'end': 274}, {'title': 'Chapter 6. Storage', 'start': 274, 'end': 337}, {'title': 'Chapter 7. Ingestion', 'start': 337, 'end': 540}, {'title': 'Chapter 9. Serving Data for\nAnalytics, Machine Learning,\nand Reverse ETL', 'start': 452, 'end': 493}, {'title': 'Chapter 10. Security and\nPrivacy', 'start': 493, 'end': 507}, {'title': 'Chapter 11. The Future of Data\nEngineering', 'start': 507, 'end': 540}]
Processing Chapter 1. Data Engineering (Pages 17 to 59)...
Processed chunk 1/42 for Chapter 1. Data Engineering
Processed chunk 2/42 for C